# Step 2 – Leakage-free cross-validation (Sections 5.1–5.5, 6)

10 × 5-fold leakage-free cross-validation of 2-Stage GeS with eleven classifiers, grouped permutation importance, KernelSHAP for the first repetition, and the leaky protocol of the IJCDS paper. Results are saved per fold, so the notebook can be re-run to resume. Run once per cohort.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

**Parameters** – change and re-run for each cohort.

In [ ]:
cohort = "GSE7390"   # one of: GSE7390, UPP, MAINZ, UNT

In [ ]:
import sys, time, pickle, numpy as np, shap
from scipy.stats import spearmanr
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedKFold
%run pipeline.ipynb

import os
REPEATS, FOLDS, PERM_R = int(os.environ.get("REP", 10)), 5, int(os.environ.get("PR", 30))
import pandas as pd
CG = list(pd.read_csv("./common_genes.csv").iloc[:, 0])
X, y, P = load(cohort, genes=CG)
Xv, genes_all = X.values, np.array(X.columns)
n = len(y)
out = {"cohort": cohort, "n": n, "classes": np.bincount(y), "folds": [], "genes_all": genes_all,
       "samples": np.array(X.index)}
BUDGET = float(os.environ.get("BUDGET", 1e9))
FD = f"./folds_{cohort}"
os.makedirs(FD, exist_ok=True)
t0 = time.time()
for rep in range(REPEATS):
    skf = StratifiedKFold(FOLDS, shuffle=True, random_state=rep)
    for k, (tr, te) in enumerate(skf.split(Xv, y)):
        fp = f"{FD}/r{rep}_k{k}.pkl"
        if os.path.exists(fp):
            continue
        if time.time() - t0 > BUDGET:
            print("BUDGET STOP", flush=True); sys.exit(0)
        r = ges_fit(Xv[tr], y[tr], seed=rep * 10 + k)
        g = r["genes"]
        Dte = apply_cuts(Xv[te][:, g], r["cuts"])
        rec = {"rep": rep, "fold": k, "te": te, "genes": genes_all[g], "n_inf": r["n_informative"],
               "n_cfs": r["n_cfs"], "n_cons": r["n_cons"], "res": {}}
        for c in CLASSIFIERS:
            m = build(c).fit(r["Dtr"], r["ytr"], r["nb"], 3)
            p = m.predict(Dte)
            rec["res"][c] = metrics(y[te], p)
            if c == "HWNB":
                hw = m
                rec["proba_hwnb"] = m.predict_proba(Dte)
                rec["pred_hwnb"] = p
        # ---- permutation importance (single gene and correlation-grouped), HWNB, held-out fold
        base = metrics(y[te], hw.predict(Dte))["macro_f1"]
        rng = np.random.default_rng(rep * 100 + k)
        mgen = len(g)
        if mgen > 1:
            rho = np.abs(spearmanr(Xv[tr][:, g]).statistic) if mgen > 2 else np.abs(np.array([[1, spearmanr(Xv[tr][:, g[0]], Xv[tr][:, g[1]]).statistic], [0, 1]]))
            if mgen == 2: rho = np.array([[1, rho[0, 1]], [rho[0, 1], 1]])
            _, lab = connected_components(rho > 0.7)
        else:
            lab = np.zeros(1, int)
        single, grouped = np.zeros(mgen), np.zeros(mgen)
        for j in range(mgen):
            drops = []
            for _ in range(PERM_R):
                D2 = Dte.copy(); D2[:, j] = rng.permutation(D2[:, j])
                drops.append(base - metrics(y[te], hw.predict(D2))["macro_f1"])
            single[j] = np.mean(drops)
        for L in np.unique(lab):
            cols = np.where(lab == L)[0]
            drops = []
            for _ in range(PERM_R):
                idx = rng.permutation(len(te)); D2 = Dte.copy(); D2[:, cols] = Dte[idx][:, cols]
                drops.append(base - metrics(y[te], hw.predict(D2))["macro_f1"])
            grouped[cols] = np.mean(drops)
        rec["perm_single"], rec["perm_grouped"], rec["groups"] = single, grouped, lab
        # ---- SHAP (first repetition only: every tumour explained once). Exact when 2^m small.
        if rep == 0:
            real_tr = apply_cuts(Xv[tr][:, g], r["cuts"])
            bg = real_tr[np.random.default_rng(k).choice(len(tr), size=min(40, len(tr)), replace=False)]
            f = lambda Z: hw.predict_proba(np.rint(Z).astype(int))
            ex = shap.KernelExplainer(f, bg)
            ns = min(2 ** mgen + 2, 4096)
            phi = ex.shap_values(Dte, nsamples=ns, silent=True)
            phi = np.array(phi)
            if phi.shape[0] == 3 and phi.ndim == 3 and phi.shape[1] == len(te):
                phi = np.transpose(phi, (1, 2, 0))  # -> n_te x m x 3
            rec["shap"] = phi
            rec["shap_base"] = np.array(ex.expected_value)
            rec["Dte"] = Dte
            rec["proba_te"] = hw.predict_proba(Dte)
        pickle.dump(rec, open(fp, "wb"))
        print(cohort, rep, k, "genes", r["n_cfs"], r["n_cons"], "HWNB F1 %.3f" % rec["res"]["HWNB"]["macro_f1"],
              "%.0fs" % (time.time() - t0), flush=True)
if os.path.exists(f"{FD}/leaky.pkl"):
    print("ALL DONE", cohort); sys.exit(0)

### Leaky protocol as in the published study: discretise + SMOTE + 2-Stage GeS on ALL data, then CV

In [ ]:
# ---- leaky protocol as in the published study: discretise + SMOTE + 2-Stage GeS on ALL data, then CV
r = ges_fit(Xv, y, seed=999)
D, Dy = r["Dtr"], r["ytr"]
leak = {c: [] for c in ["HWNB", "NB", "CWNB"]}
for rep in range(REPEATS):
    skf = StratifiedKFold(FOLDS, shuffle=True, random_state=rep)
    for tr, te in skf.split(D, Dy):
        for c in leak:
            m = build(c).fit(D[tr], Dy[tr], r["nb"], 3)
            leak[c].append(metrics(Dy[te], m.predict(D[te])))
leaky = {"genes": genes_all[r["genes"]], "n_cfs": r["n_cfs"], "n_cons": r["n_cons"], "res": leak}
pickle.dump(leaky, open(f"{FD}/leaky.pkl", "wb")); pickle.dump({k: out[k] for k in ["cohort","n","classes","genes_all","samples"]}, open(f"{FD}/meta.pkl","wb"))
print("DONE", cohort, "%.0fs" % (time.time() - t0))